# 03 — Churn & demand baselines

Honest baselines for Model Insights — not production models.

**Questions**
- Without Recency (leakage risk), can Frequency / Monetary / Tenure separate Churned90?
- How does LogisticRegression compare to a quick GBM experiment?
- Is seasonal-naive (lag-12) enough as a demand stub on ~25 months?


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path('..')
GOLD = ROOT / 'data' / 'gold'
OUT = Path('outputs')
REPORTS = ROOT / 'reports'
OUT.mkdir(parents=True, exist_ok=True)

feat = pd.read_csv(GOLD / 'ml_customer_features.csv')
feat = feat[feat['CustomerKey'] > 0].copy()
print('customers (registered):', len(feat))


In [ ]:
churn_cols = ['Frequency', 'Monetary', 'AvgOrderValue', 'AvgBasketSize', 'TenureDays']
Xc = feat[churn_cols].fillna(0)
yc = feat['Churned90'].astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.25, random_state=42, stratify=yc)
print(f'train={len(Xtr)}  test={len(Xte)}  base_rate={yc.mean():.3f}')

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)
logit = LogisticRegression(max_iter=500, random_state=42)
logit.fit(Xtr_s, ytr)
proba_l = logit.predict_proba(Xte_s)[:, 1]
pred_l = (proba_l >= 0.5).astype(int)
lr_acc = float(accuracy_score(yte, pred_l))
lr_auc = float(roc_auc_score(yte, proba_l))
print(f'LogisticRegression  accuracy={lr_acc:.4f}  AUC={lr_auc:.4f}')

gbc = GradientBoostingClassifier(random_state=42, max_depth=3, n_estimators=120, learning_rate=0.08)
gbc.fit(Xtr, ytr)
proba_g = gbc.predict_proba(Xte)[:, 1]
pred_g = (proba_g >= 0.5).astype(int)
gb_acc = float(accuracy_score(yte, pred_g))
gb_auc = float(roc_auc_score(yte, proba_g))
print(f'GradientBoosting    accuracy={gb_acc:.4f}  AUC={gb_auc:.4f}')


In [ ]:
churn_metrics = {
    'target': 'Churned90 (no purchase in last 90 days of observation window)',
    'features': churn_cols,
    'n_train': int(len(Xtr)), 'n_test': int(len(Xte)),
    'base_rate': round(float(yc.mean()), 4),
    'logistic_regression': {'accuracy': round(lr_acc, 4), 'auc': round(lr_auc, 4)},
    'gradient_boosting': {
        'accuracy': round(gb_acc, 4), 'auc': round(gb_auc, 4),
        'feature_importance': {c: round(float(v), 4) for c, v in zip(churn_cols, gbc.feature_importances_)},
        'note': 'optional comparison — not the primary baseline',
    },
    'limitations': [
        'Label is window-end inactivity, not a true future holdout churn event.',
        'Recency intentionally excluded from features to avoid trivial leakage.',
        'Guest checkouts excluded; UK-heavy mix may not transfer to other markets.',
    ],
}

rep = feat[feat['Frequency'] >= 2].copy()
Xr = rep[['Frequency', 'TenureDays', 'AvgBasketSize', 'RecencyDays']].fillna(0)
yr = rep['AvgOrderValue']
Xtr2, Xte2, ytr2, yte2 = train_test_split(Xr, yr, test_size=0.25, random_state=42)
gbr = GradientBoostingRegressor(random_state=42, max_depth=3, n_estimators=120, learning_rate=0.08)
gbr.fit(Xtr2, ytr2)
pred_r = gbr.predict(Xte2)
naive = np.full_like(yte2, ytr2.mean(), dtype=float)
aov_metrics = {
    'target': 'AvgOrderValue (customers with 2+ orders)',
    'features': list(Xr.columns),
    'n_train': int(len(Xtr2)), 'n_test': int(len(Xte2)),
    'mae': round(float(mean_absolute_error(yte2, pred_r)), 2),
    'r2': round(float(r2_score(yte2, pred_r)), 4),
    'naive_mean_mae': round(float(mean_absolute_error(yte2, naive)), 2),
    'mean_aov_test': round(float(yte2.mean()), 2),
}

monthly = pd.read_csv(GOLD / 'mart_monthly.csv').sort_values('YearMonth')
monthly['RevLag1'] = monthly['Revenue'].shift(1)
monthly['RevLag12'] = monthly['Revenue'].shift(12)
eval12 = monthly.dropna(subset=['RevLag12'])
mae_m = float(mean_absolute_error(eval12['Revenue'], eval12['RevLag12']))
mape = float(np.mean(np.abs((eval12['Revenue'] - eval12['RevLag12']) / eval12['Revenue'])) * 100)
demand_metrics = {
    'model': 'Seasonal naive (lag-12 months)',
    'mae_revenue': round(mae_m, 2), 'mape_pct': round(mape, 2),
    'n_eval_months': int(len(eval12)),
}
out = {'churn': churn_metrics, 'aov_baseline': aov_metrics, 'demand_baseline': demand_metrics}
(REPORTS / 'model_metrics.json').write_text(json.dumps(out, indent=2))
print('Demand MAPE', demand_metrics['mape_pct'], '%')
out


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
ax.hist(proba_l[yte.values == 0], bins=20, alpha=0.65, label='Active', density=True)
ax.hist(proba_l[yte.values == 1], bins=20, alpha=0.65, label='Churned90', density=True)
ax.set_title(f'LR churn scores  ·  AUC {lr_auc:.3f}')
ax.set_xlabel('Predicted P(churn)')
ax.legend()
imps = churn_metrics['gradient_boosting']['feature_importance']
ax2 = axes[1]
ax2.barh(list(imps.keys()), list(imps.values()), color='#F2C811')
ax2.set_title('GBM feature importance (experiment)')
fig.tight_layout()
fig.savefig(OUT / 'churn_baseline.png', dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly['YearMonth'], monthly['Revenue'] / 1e6, marker='o', markersize=3, label='Actual')
ax.plot(monthly['YearMonth'], monthly['RevLag12'] / 1e6, linestyle='--', label='Seasonal naive')
ax.set_title('Monthly revenue vs seasonal-naive baseline')
ax.tick_params(axis='x', labelrotation=45, labelsize=7)
ax.set_ylabel('£M')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'demand_baseline.png', dpi=140)
plt.show()
